# A tiny transformer in numpy

Nine words, 64 numbers per word, random untrained weights (the way a model starts before training).

In [1]:
import numpy as np

D, L = 64, 48
NOTE = ["Tom", "owes", "Ana", "fifty", "dollars", ".", "Who", "pays", "?"]
SWAP = ["Ana", "owes", "Tom", "fifty", "dollars", ".", "Who", "pays", "?"]
SEEDS = range(20)


def layer_norm(x):
    x = x - x.mean(-1, keepdims=True)
    return x / np.sqrt((x ** 2).mean(-1, keepdims=True) + 1e-6)


def softmax(s):
    e = np.exp(s - s.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


def positions(n):                       # sine and cosine waves, one pattern per position (Vaswani et al., 2017)
    p, i = np.arange(n)[:, None], np.arange(D // 2)[None]
    ang = p / 10000 ** (2 * i / D)
    out = np.zeros((n, D))
    out[:, 0::2], out[:, 1::2] = np.sin(ang), np.cos(ang)
    return out


def weights(seed, out_std=0.02):
    r = np.random.default_rng(seed)
    g = lambda shape, s: r.normal(0, s, shape)
    return [dict(q=g((D, D), D ** -0.5), k=g((D, D), D ** -0.5), v=g((D, D), D ** -0.5), o=g((D, D), out_std),
                 f1=g((D, 4 * D), D ** -0.5), f2=g((4 * D, D), out_std)) for _ in range(L)]


def attention(x, b, causal=False, sharp=4.0):
    s = (x @ b["q"]) @ (x @ b["k"]).T / np.sqrt(D) * sharp      # query . key
    if causal:
        s = s + np.triu(np.full(s.shape, -1e9), 1)               # look back only
    w = softmax(s)                                               # shares, each row adds up to 1
    return w @ (x @ b["v"]) @ b["o"], w                          # blend of values


def feed_forward(x, b):
    return np.maximum(x @ b["f1"], 0) @ b["f2"]                  # spread out, switch off, squeeze back


def embed(words, seed, with_positions):
    r = np.random.default_rng(1000 + seed)
    table = {w: r.normal(0, 1, D) for w in sorted(set(NOTE))}
    x = np.array([table[w] for w in words])
    return x + positions(len(words)) if with_positions else x


def n_diff(a, b):
    return int((np.abs(a - b) > 1e-9).sum())

## One round of attention: the sentence vs the same sentence with the names swapped

Numbers that differ at the last word ("?"), out of 64.

In [2]:
def one_round(seed, with_positions, causal):
    b = weights(seed, out_std=D ** -0.5)[0]
    oa, wa = attention(layer_norm(embed(NOTE, seed, with_positions)), b, causal)
    ob, _ = attention(layer_norm(embed(SWAP, seed, with_positions)), b, causal)
    return n_diff(oa[-1], ob[-1]), wa[-1]


for with_positions in (False, True):
    for causal in (False, True):
        n, shares = one_round(0, with_positions, causal)
        print(f"positions: {with_positions!s:5}  look back only: {causal!s:5}  ->  {n} of {D} differ")

_, shares = one_round(0, False, False)
print("\nshares at '?':", {w: round(float(s), 2) for w, s in zip(NOTE, shares)})

positions: False  look back only: False  ->  0 of 64 differ
positions: False  look back only: True   ->  0 of 64 differ
positions: True   look back only: False  ->  64 of 64 differ
positions: True   look back only: True   ->  64 of 64 differ

shares at '?': {'Tom': 0.19, 'owes': 0.01, 'Ana': 0.26, 'fifty': 0.0, 'dollars': 0.0, '.': 0.14, 'Who': 0.0, 'pays': 0.0, '?': 0.39}


## The same over 20 random set-ups

In [3]:
runs = [(s, c) for s in SEEDS for c in (False, True)]
no_pos_same = sum(one_round(s, False, c)[0] == 0 for s, c in runs)
pos_all_differ = sum(one_round(s, True, c)[0] == D for s, c in runs)
print(f"no positions: identical at '?' in {no_pos_same} of {len(runs)} runs")
print(f"with positions: all {D} numbers differ in {pos_all_differ} of {len(runs)} runs")

no positions: identical at '?' in 40 of 40 runs
with positions: all 64 numbers differ in 40 of 40 runs


## 48 blocks: each block replaces the numbers, or adds to them (residual connection)

How different the words still are from each other, as a share of how different they were at the start.

In [4]:
def spread(x):
    u = x / np.linalg.norm(x, axis=1, keepdims=True)
    d = np.linalg.norm(u[:, None] - u[None], axis=-1)
    return float(d.sum() / (len(x) ** 2 - len(x)))


def stack(seed):
    W, x0 = weights(seed), embed(NOTE, seed, True)
    s0, x, y, plain, residual = spread(x0), x0.copy(), x0.copy(), [], []
    for b in W:
        x = layer_norm(attention(x, b)[0])          # replaces
        x = layer_norm(feed_forward(x, b))
        y = y + attention(layer_norm(y), b)[0]      # adds
        y = y + feed_forward(layer_norm(y), b)
        plain.append(spread(x) / s0); residual.append(spread(y) / s0)
    return plain, residual


AT = [1, 2, 3, 4, 6, 12, 24, 48]
plain, residual = stack(0)
print("after block:", AT)
print("replace:    ", [f"{plain[i - 1]:.0%}" for i in AT])
print("add:        ", [f"{residual[i - 1]:.0%}" for i in AT])

at48 = [stack(s)[1][-1] for s in SEEDS]
print(f"\nadd, block 48, 20 set-ups: {min(at48):.0%} to {max(at48):.0%}")

after block: [1, 2, 3, 4, 6, 12, 24, 48]
replace:     ['64%', '24%', '2%', '0%', '0%', '0%', '0%', '0%']
add:         ['100%', '100%', '100%', '101%', '102%', '98%', '96%', '93%']

add, block 48, 20 set-ups: 70% to 94%


## Control: attention only vs feed-forward only (each block replaces), median of 20 set-ups

In [5]:
def single(seed, step):
    W, x = weights(seed), embed(NOTE, seed, True)
    s0, out = spread(x), []
    for b in W:
        x = layer_norm(attention(x, b)[0] if step == "attention" else feed_forward(x, b))
        out.append(spread(x) / s0)
    return out


medians = {}
for step in ("attention", "feed_forward"):
    runs = np.array([single(s, step) for s in SEEDS])
    medians[step] = [float(np.median(runs[:, i - 1])) for i in AT]
    print(f"{step:12}", [f"{m:.0%}" for m in medians[step]])

attention    ['78%', '43%', '6%', '0%', '0%', '0%', '0%', '0%']
feed_forward ['85%', '74%', '64%', '57%', '47%', '34%', '17%', '7%']
